In [56]:
import math
import random
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from time import sleep
from collections import deque, defaultdict
from itertools import count
from typing import Dict

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from importnb import Notebook
with Notebook():
    from LabLatencyModel import LatencyModel, MultiDULatencyModel
    from LabCacheEngine import CacheEngineEnv, CacheUnitMapper
    from LabUserTileRequest import UserTileRequestEvents
    from LabEnvWrapper import EnvWrapper

import sys
sys.path.append('/home/eduardo/Workspace/CacheVideoPredict360/Sources')
# sys.path.append(r'c:\Users\es25591\Workspace\CacheVideoPredict360\Sources')
from Common.Utils import save_training_results


In [57]:
# --- 1. CONFIGURATION & HYPERPARAMETERS (Section VII-B) ---
class Config:
    n_episodes: int = 300
    n_nodes: int = 3
    n_users: int = 1
    step_size: float = 10.00
    arrival_rate: float = 10.0  # users per second
    alpha: float = 0.5
    n_videos: int = 500
    n_gops: int = 30
    n_layers: int = 2
    n: int = 4
    m: int = 3
    n_tiles: int = n * m
    tiles_per_viewport: int = 4

    base_tile_b = 2e6 / n_tiles
    enh_tile_b = 15e6 / n_tiles

    max_capacity: float = 500e6  # 500 MB
    cache_capacity_percent: float = 0.1  # 10% of the total video size
    cache_size: int = int(n_videos * cache_capacity_percent)
    cache_capacity_b: float = (
        n_gops * n_tiles * base_tile_b +
        n_gops * tiles_per_viewport * enh_tile_b
    ) * cache_size

    # Hyperparameters for RL
    epsilon_start: float = 1.0
    epsilon_min: float = 0.005
    epsilon_decay: float = 0.987
    gamma: float = 0.99
    learning_rate: float = 1e-3
    batch_size: int = 32
    capacity: int = 10000
    window_len: int = 3  # LSTM sequence length (history window)

    h_short: int = 300   # sliding windows for popularity (Section VI-A)
    h_long: int = 1000

    r_base: float = 30.0 # PSNR reward for base layer (Section VI-C)
    r_enh: float = 10.0  # PSNR reward for enhancement layer
    penalty: float = 0.0 # fetch penalty (implicit in paper)

    # CPT parameters
    theta: float = 0.5
    lam: float = 3.7183

    @property
    def state_dim(self) -> int:
        # 10*C + 2 = (2C + 2Ck) * 2 + 2 (Section VI-A)
        return 10 * self.cache_size + 2

    @property
    def action_dim(self) -> int:
        # |A| = 5C + 1 (Section VI-B)
        return 5 * self.cache_size + 1

In [58]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class DQN(nn.Module):
    def __init__(self, state_dim: int, action_dim: int):
        super().__init__()
        hidden = action_dim  # = 5C + 1
        self.fc1 = nn.Linear(state_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.out = nn.Linear(hidden, action_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)  # linear

class ReplayBuffer:
    def __init__(self, capacity: int = 2000):
        self.memory = deque(maxlen=capacity)
    def push(self, s, a, r, ns, d):
        self.memory.append((s, a, r, ns, d))
    def sample(self, batch_size: int):
        return random.sample(self.memory, batch_size)
    def __len__(self):
        return len(self.memory)

class DQNAgent:
    def __init__(self, cfg: Config):
        self.state_dim = cfg.state_dim
        self.action_dim = cfg.action_dim
        self.epsilon = 0.05
        self.gamma = 0.6
        self.batch_size = 32
        self.buffer = ReplayBuffer(2000)
        self.policy_net = DQN(self.state_dim, self.action_dim).to(device)
        self.target_net = DQN(self.state_dim, self.action_dim).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=1e-3)
        self.loss_fn = nn.MSELoss()
        self.nb_interval = 200  # train every 200 requests

    def select_action(self, state_vec: np.ndarray) -> int:
        if random.random() < self.epsilon:
            return random.randrange(self.action_dim)
        with torch.no_grad():
            s = torch.from_numpy(state_vec).float().to(device).unsqueeze(0)
            q = self.policy_net(s)
            return int(q.argmax().item())

    def remember(self, s, a, r, ns, done):
        self.buffer.push(s, a, r, ns, done)

    def train_step(self):
        if len(self.buffer) < self.batch_size:
            return
        batch = self.buffer.sample(self.batch_size)
        s, a, r, ns, d = zip(*batch)
        s = torch.tensor(np.stack(s), dtype=torch.float32).to(device)
        ns = torch.tensor(np.stack(ns), dtype=torch.float32).to(device)
        a = torch.tensor(a, dtype=torch.int64).to(device)
        r = torch.tensor(r, dtype=torch.float32).to(device)
        d = torch.tensor(d, dtype=torch.float32).to(device)

        q = self.policy_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            q_next = self.target_net(ns).max(1)[0]
            target = r + self.gamma * q_next * (1.0 - d)

        loss = self.loss_fn(q, target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

    def update_target(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())

In [ ]:
class FeatureAdapter:
    def __init__(self, env: CacheEngineEnv, cfg: Config):
        self.env = env
        self.cfg = cfg

        self.video_hist_short = deque(maxlen=cfg.h_short)
        self.video_hist_long = deque(maxlen=cfg.h_long)
        self.tile_hist_short = deque(maxlen=cfg.h_short)
        self.tile_hist_long = deque(maxlen=cfg.h_long)

        self.video_freq_short = defaultdict(int)
        self.video_freq_long = defaultdict(int)
        self.tile_freq_short = defaultdict(int)
        self.tile_freq_long = defaultdict(int)

    def reset_history(self):
        queues = (
            self.video_hist_short,
            self.video_hist_long,
            self.tile_hist_short,
            self.tile_hist_long,
        )
        freqs = (
            self.video_freq_short,
            self.video_freq_long,
            self.tile_freq_short,
            self.tile_freq_long,
        )
        for q in queues:
            q.clear()
        for f in freqs:
            f.clear()

    def update_history(self, request: Dict):
        vid = request["video"]
        
        self._update_window(self.video_hist_short, self.video_freq_short, vid)
        self._update_window(self.video_hist_long, self.video_freq_long, vid)

        tiles = request["viewport"]
        tiles = tuple(tiles.tolist()) if len(tiles) > 0 else None
        
        self._update_window(self.tile_hist_short, self.tile_freq_short, tiles)
        self._update_window(self.tile_hist_long, self.tile_freq_long, tiles)
        
    def _update_window(self, hist_queue: deque, freq_dict: Dict, item):
        if item is None:
            return
        
        if len(hist_queue) == hist_queue.maxlen:
            old_item = hist_queue.popleft()
            freq_dict[old_item] -= 1
            if freq_dict[old_item] == 0:
                del freq_dict[old_item]
        hist_queue.append(item)
        freq_dict[item] += 1

In [60]:
class NetworkAdapter:
    def __init__(self, env: EnvWrapper, feature_adapter: FeatureAdapter, cfg: Config):
        self.env = env
        self.cfg = cfg
        self.features = feature_adapter
        self.capacity = env.mec_cache.unit_mapper.max_units if env.mec_cache.unit_mapper is not None else 0
        self.n_features = 2 * (self.capacity * 5 + 1)

        self.capacity = int(
            self.cfg.n_videos * self.cfg.cache_capacity_percent * self.cfg.n_gops * self.cfg.n_tiles +
            self.cfg.n_videos * self.cfg.cache_capacity_percent * self.cfg.n_gops * self.cfg.tiles_per_viewport            
        )
        print(f"NetworkAdapter initialized with capacity: {self.capacity}")
        
    def _rank_cached_videos(self, bitmap: np.ndarray) -> list[int]:
        cached_mask = np.any(bitmap[:, 0, :, :], axis=(1, 2))
        cached_vids = np.where(cached_mask)[0].tolist()
        ranked = sorted(
            cached_vids,
            key=lambda vid: self.features.video_freq_long.get(vid, 0),
            reverse=True
        )
        pad = [-1] * (self.capacity - len(ranked))
        return ranked + pad
    
    def _cached_viewport_tiles(self, bitmap: np.ndarray, vid: int, k: int) -> list[int]:
        if vid == -1:
            return [-1] * k
        tile_mask = np.any(bitmap[vid, 1, :, :], axis=1)
        tiles = np.flatnonzero(tile_mask).tolist()
        tiles.extend([-1] * max(0, k - len(tiles)))
        return tiles[:k]
    
    def build_observation(self, request: Dict) -> np.ndarray:
        cache_bitmap = self.env.mec_cache.get_cache_bitmap()
        ranked_cached = self._rank_cached_videos(cache_bitmap)
        
        k = self.env.mec_cache.get_viewport_tile_budget()
        
        n_features = 2 * (self.capacity * 5 + 1)

        obs_long = np.zeros(n_features, dtype=np.float32)
        obs_short = np.zeros(n_features, dtype=np.float32)

        for idx, vid in enumerate(ranked_cached):
            if vid == -1:
                obs_short[idx] = 0
                obs_long[idx] = 0
            else:
                obs_long[idx] = self.features.video_freq_long.get(vid, 0) / self.features.cfg.h_long
                obs_short[idx] = self.features.video_freq_short.get(vid, 0) / self.features.cfg.h_short

            tiles = self._cached_viewport_tiles(cache_bitmap, vid, k)
            base = self.capacity + idx * k

            for off, tile_id in enumerate(tiles):
                if tile_id == -1:
                    obs_short[base + off] = 0
                    obs_long[base + off] = 0
                else:
                    obs_long[base + off] = self.features.tile_freq_long.get(tile_id, 0) / self.features.cfg.h_long
                    obs_short[base + off] = self.features.tile_freq_short.get(tile_id, 0) / self.features.cfg.h_short

        vid = request["video"]
        obs_long[-1] = self.features.video_freq_long.get(vid, 0) / self.features.cfg.h_long
        obs_short[-1] = self.features.video_freq_short.get(vid, 0) / self.features.cfg.h_short

        return np.concatenate([obs_long, obs_short], axis=0)
    

    def evict_video(self, bitmap: np.ndarray, v: int):
        for layer, tile_id, gop_id in np.argwhere(bitmap[v] == 1):
            key = (v, int(layer), int(tile_id), int(gop_id))
            self.env.mec_cache.policy.remove(key)
            bitmap[v, layer, tile_id, gop_id] = 0
                    
    def apply_action(self, action_idx: int, request: Dict) -> Dict:
        """
        Implements the paper’s action space:
          - A1 (size C+1): when base is not cached. a0 = no-op; a_i evicts the i-th cached video and caches the requested one.
          - A2 (size k+1): when base is cached but viewport differs. a0 = no-op; a_j replaces the j-th cached enh tile with the j-th requested tile.
        """
        bitmap = self.env.mec_cache.get_cache_bitmap()
        
        vid = request["video"]
        gop = request["gop"]
        viewport = request["viewport"]

        C = self.capacity
        k = self.env.mec_cache.get_viewport_tile_budget() # 4

        base_cached = np.any(bitmap[vid, 0, :, :])
        enh_cached = all(bitmap[vid, 1, t, gop] == 1 for t in viewport)

        # Block A1: video not cached in base
        if not base_cached:
            if action_idx == 0:
                return  # no-op
            if 1 <= action_idx <= C:
                ranked = self._rank_cached_videos(bitmap)
                victim = ranked[action_idx - 1]
                if victim != -1:
                    self.evict_video(bitmap, victim)
                # Cache requested video (base + provided tiles structure)
                self.env.mec_cache.cache_new_video(vid, gop, request.get("tiles", []))
            return

        # Block A2: base cached; check viewport
        if enh_cached:
            return  # already fully cached, no action

        # offset into A2
        a2_idx = action_idx - (C + 1)
        if a2_idx == 0:
            return  # no-op
        if 1 <= a2_idx <= k:
            j = a2_idx - 1
            if j < len(viewport):
                tile_id = viewport[j]
                key = (vid, 1, tile_id, gop)
                tile_size = int(self.env.mec_cache.tile_size_bytes[1])
                evicted = self.env.mec_cache.policy.put(key, None, tile_size)
                bitmap[vid, 1, tile_id, gop] = 1
                for e_vid, e_layer, e_tile, e_gop in evicted:
                    bitmap[e_vid, e_layer, e_tile, e_gop] = 0


In [61]:
if __name__ == "__main__":
    print("--- Starting DRL Caching System ---")

    # 1. Load Configuration
    cfg = Config()
    
    # 2. Initialize Environment
    du_caches = []

    unit_mapper = CacheUnitMapper(
        cache_capacity_mb=cfg.cache_capacity_b / 1e6,
        num_gops=cfg.n_gops,
        num_tiles=cfg.n_tiles,
        viewport_tiles=4,  # assuming viewport with 4 tiles
        base_tile_mb=2e6 / 1e6 / cfg.n_tiles,
        enh_tile_mb=15e6 / 1e6 / cfg.n_tiles
    )
    
    mec_cache = CacheEngineEnv(
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n_gops=cfg.n_gops,
        cache_capacity=cfg.cache_capacity_b,
        # policy=SvcLruPolicy(max_size=max_capacity)
        unit_mapper=unit_mapper
    )

    # Initialize User Environment
    users_env = UserTileRequestEvents(
        n_nodes=cfg.n_nodes,
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_gops=cfg.n_gops,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n=cfg.n,
        m=cfg.m,
        users_viewport_tiles=None,
        requested_videos=None,
        users_arrivals=None,
        arrival_rate=cfg.arrival_rate,
        alpha=cfg.alpha
    )

    P = cfg.n_nodes; max_U = cfg.n_users
    lat_model = MultiDULatencyModel(
        P=P, 
        max_U=max_U,
        R_M_D=80e6,  # 640 Mbps -> 80e6 B/s
        R_C_M=125e6, # 1 Gbps -> 125e6 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 20e6, dtype=float),    # 160 Mbps -> 20e6 B/s
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float), # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,  # 1 ms
        mec_fixed_delay=0.005, # 5 ms
        cloud_fixed_delay=0.1  # 100 ms
    )

    env = EnvWrapper(
        n=cfg.n,
        n_layers=cfg.n_layers,
        users_env=users_env, 
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=lat_model,
        theta=cfg.theta,
        lam=cfg.lam
    )

    # 3. Initialize Agent
    agent = DQNAgent(cfg)

    obs, info = env.reset()
    feature_adapter = FeatureAdapter(env, cfg)
    net_adapter = NetworkAdapter(env, feature_adapter, cfg)

    print(
        f"->> "
        f"{mec_cache.unit_mapper.max_units} "
        f"{mec_cache.max_capacity}"
    )
    
    for step in count():
        reqs_state = info['users_requests']
        active_users = [
            req  for req in reqs_state if req['gop'] < cfg.n_gops
        ]
    
        for req in active_users:
            feature_adapter.update_history(req)

        actions = []
        for user_req in active_users:
            video = user_req['video']
            tiles = user_req['tiles']
            gop = user_req['gop']

            state_vec = net_adapter.build_observation(user_req)
            # action_idx = np.random.randint(0, net_adapter.capacity * 5 + 1)
            action_idx = agent.select_action(state_vec)
            net_adapter.apply_action(action_idx, user_req)

            actions.append({
                "user": user_req["u"],
                "gop": gop,
                "video": video,
                "action": action_idx
            })

        obs, rewards, done, info = env.step(actions)

        step_reward = float(rewards) / max(1, len(active_users))

        # --- TRAINING / HISTORY UPDATE ---
        reqs_next_state = info['users_requests']

        for user_req in active_users:


            agent.remember(state_vec, action_idx, step_reward, next_state_vec, done_flag)
            if step % agent.nb_interval == 0:
                agent.train_step()
                agent.update_target()

        print(f"Step {step}, Active Users: {len(active_users)}")
        print(f"Request State: {reqs_state}")
        print(f"Action: {actions}")
        print(f"Next Request State: {reqs_next_state}")
        print("-----")

        if done:
            break

--- Starting DRL Caching System ---
NetworkAdapter initialized with capacity: 24000
->> 50 10500000000.0
Step 0, Active Users: 0
Request State: []
Action: []
Next Request State: []
-----
Step 1, Active Users: 0
Request State: []
Action: []
Next Request State: []
-----
Step 2, Active Users: 0
Request State: []
Action: []
Next Request State: []
-----
Step 3, Active Users: 0
Request State: []
Action: []
Next Request State: []
-----
Step 4, Active Users: 0
Request State: []
Action: []
Next Request State: []
-----
Step 5, Active Users: 0
Request State: []
Action: []
Next Request State: []
-----
Step 6, Active Users: 0
Request State: []
Action: []
Next Request State: []
-----
Step 7, Active Users: 0
Request State: []
Action: []
Next Request State: []
-----
Step 8, Active Users: 0
Request State: []
Action: []
Next Request State: []
-----
Step 9, Active Users: 0
Request State: []
Action: []
Next Request State: []
-----
Step 10, Active Users: 0
Request State: []
Action: []
Next Request State: [

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x480004 and 502x251)